In [ ]:
# IMPORT THƯ VIỆN
 
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
 
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import timm
 
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score, accuracy_score
)
from pathlib import Path
 
print("Import hoàn tất.")
print(f"PyTorch : {torch.__version__} | CUDA: {torch.cuda.is_available()}")
 

In [ ]:
# CẤU HÌNH
# Trỏ đến 2 best model đã train, dataset split, và thư mục output.
 
RESNET_PATH   = "/kaggle/input/notebooks/dunnguynvn/train-resnet50-ver2/resnet50_output/resnet50_best.pth"
SWIN_PATH     = "/kaggle/input/notebooks/dunnguynvn/trainswinver2/swin_output/swin_best.pth"
SPLIT_PATH    = "/kaggle/input/datasets/dunnguynvn/blood-data-l2/dataset_split"
OUTPUT_DIR    = "/kaggle/working/ensemble_output"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
 
CLASSES = [
    "BA", "BNE", "EO", "ERB", "IG", "LY", "MMY",
    "MO", "MY", "MYELOBLAST", "PLATELET", "PMY", "SNE"
]
NUM_CLASSES = len(CLASSES)
 
# Nhóm lớp khó dòng tủy bào có hình thái tương tự nhau
# Swin mạnh hơn ResNet ở cả 3 lớp này -> tăng trọng số Swin
HARD_CLASSES = ["MY", "MMY", "PMY"]
 
IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_WORKERS = 2
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
# Kết quả đã biết từ 2 notebook trước
RESNET_RESULTS = {"accuracy": 0.9401, "f1_macro": 0.9220}
SWIN_RESULTS   = {"accuracy": 0.9449, "f1_macro": 0.9260}
 
# F1 per-class từ kết quả thực tế của 2 model đơn
RESNET_F1_PER_CLASS = {
    "BA": 0.9954, "BNE": 0.8656, "EO": 0.9948, "ERB": 0.9933,
    "IG": 1.0000, "LY": 0.9828, "MMY": 0.7960, "MO": 0.9854,
    "MY": 0.7965, "MYELOBLAST": 1.0000, "PLATELET": 0.9954,
    "PMY": 0.7368, "SNE": 0.8435
}
SWIN_F1_PER_CLASS = {
    "BA": 0.9954, "BNE": 0.8639, "EO": 1.0000, "ERB": 1.0000,
    "IG": 0.9831, "LY": 0.9873, "MMY": 0.8000, "MO": 0.9853,
    "MY": 0.8120, "MYELOBLAST": 1.0000, "PLATELET": 1.0000,
    "PMY": 0.7568, "SNE": 0.8548
}
 
print(f"Device : {DEVICE}")
print(f"Output : {OUTPUT_DIR}")

In [ ]:
# DATA TRANSFORMS & DATALOADER
# Chỉ cần val và test loader cho inference
 
transform_val = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])
 
val_dataset  = datasets.ImageFolder(
    root=os.path.join(SPLIT_PATH, "val"),  transform=transform_val)
test_dataset = datasets.ImageFolder(
    root=os.path.join(SPLIT_PATH, "test"), transform=transform_val)
 
val_loader  = DataLoader(val_dataset,  batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=NUM_WORKERS)
 
print(f"Val  : {len(val_dataset)} ảnh")
print(f"Test : {len(test_dataset)} ảnh")

In [ ]:
# LOAD HAI MÔ HÌNH ĐÃ TRAIN
# Load ResNet50 và Swin-S.
# Chuyển sang eval() mode — không cập nhật gradient khi inference.
 
def build_resnet50(num_classes):
    model = models.resnet50(weights=None)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.2),
        nn.Linear(512, num_classes)
    )
    return model
 
def build_swin(num_classes):
    backbone = timm.create_model(
        "swin_small_patch4_window7_224",
        pretrained=False,
        num_classes=0,
        global_pool='avg'
    )
    in_features = backbone.num_features
 
    class SwinClassifier(nn.Module):
        def __init__(self, backbone, in_features, num_classes):
            super().__init__()
            self.backbone = backbone
            self.head = nn.Sequential(
                nn.Dropout(p=0.3),
                nn.Linear(in_features, 384),
                nn.ReLU(inplace=True),
                nn.Dropout(p=0.2),
                nn.Linear(384, num_classes)
            )
        def forward(self, x):
            return self.head(self.backbone(x))
 
    return SwinClassifier(backbone, in_features, num_classes)
 
resnet_model = build_resnet50(NUM_CLASSES).to(DEVICE)
swin_model   = build_swin(NUM_CLASSES).to(DEVICE)
 
ckpt_r = torch.load(RESNET_PATH, map_location=DEVICE)
ckpt_s = torch.load(SWIN_PATH,   map_location=DEVICE)
resnet_model.load_state_dict(ckpt_r["model_state"])
swin_model.load_state_dict(ckpt_s["model_state"])
 
resnet_model.eval()
swin_model.eval()
 
print(f"ResNet50 loaded (epoch={ckpt_r['epoch']}, val_acc={ckpt_r['val_acc']:.4f})")
print(f"Swin-S   loaded (epoch={ckpt_s['epoch']}, val_acc={ckpt_s['val_acc']:.4f})")

In [ ]:
# INFERENCE VỚI TEMPERATURE SCALING
# Temperature Scaling: chia logits cho T trước Softmax.
#   T > 1: làm mềm xác suất tốt cho ensemble
#   T = 1: Softmax thông thường
# ResNet50 dùng T=1.2 vì CNN thường overconfident hơn Transformer.
# Swin dùng T=1.0 vì attention-based model đã calibrated tốt hơn.
 
def get_probabilities(model, loader, T=1.0):
    """Inference toàn bộ loader, trả về xác suất và nhãn thực (có temperature scaling)."""
    all_probs  = []
    all_labels = []
 
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            logits = model(images)
            # Temperature scaling: T>1 làm mềm xác suất, giảm overconfidence
            probs  = torch.softmax(logits / T, dim=1)
            all_probs.append(probs.cpu().numpy())
            all_labels.extend(labels.numpy())
 
    return np.vstack(all_probs), np.array(all_labels)
 
print("Inference trên tập VAL...")
resnet_val_probs, val_labels = get_probabilities(resnet_model, val_loader, T=1.2)
swin_val_probs,   _          = get_probabilities(swin_model,   val_loader, T=1.0)
 
print("Inference trên tập TEST...")
resnet_test_probs, test_labels = get_probabilities(resnet_model, test_loader, T=1.2)
swin_test_probs,   _           = get_probabilities(swin_model,   test_loader, T=1.0)
 
print(f"Val probs shape  : {resnet_val_probs.shape}")
print(f"Test probs shape : {resnet_test_probs.shape}")

In [ ]:
# GRID SEARCH TRỌNG SỐ TRÊN TẬP VAL
# Grid search mịn hơn (step 0.05): w_swin = 0.05 -> 0.95.
# Adaptive weighting cho nhóm lớp khó (MY, MMY, PMY):
#   Dùng cố định 0.3/0.7 (ResNet/Swin) vì Swin đã chứng minh mạnh hơn
 
print("\n" + "=" * 60)
print("GRID SEARCH TRỌNG SỐ TRÊN TẬP VAL (IMPROVED)")
print("=" * 60)
print(f"  {'w_resnet':>10} {'w_swin':>10} {'F1 Macro':>10} {'Accuracy':>10}")
print(f"  {'-'*45}")
 
weight_results = []
best_f1  = 0.0
best_w_r = 0.5
best_w_s = 0.5
 
# Grid mịn hơn: step 0.05 → 19 điểm thay vì 9
weights = [round(i / 100, 2) for i in range(5, 96, 5)]
 
for w_s in weights:
    w_r = round(1.0 - w_s, 2)
 
    # Ensemble cơ bản: weighted average toàn bộ lớp
    prob_combined = w_r * resnet_val_probs + w_s * swin_val_probs
 
    # Adaptive weighting cho nhóm lớp khó:
    # 0.3/0.7 được chọn dựa trên bằng chứng thực nghiệm:
    # Swin > ResNet ở MY (+0.0155), PMY (+0.0200), MMY (+0.0040)
    for cls in HARD_CLASSES:
        idx = CLASSES.index(cls)
        prob_combined[:, idx] = (
            0.3 * resnet_val_probs[:, idx] +
            0.7 * swin_val_probs[:, idx]
        )
 
    preds  = prob_combined.argmax(axis=1)
    f1_mac = f1_score(val_labels, preds, average='macro', zero_division=0)
    acc    = accuracy_score(val_labels, preds)
 
    weight_results.append({
        "w_resnet": w_r, "w_swin": w_s,
        "f1_macro": round(f1_mac, 4),
        "accuracy": round(acc, 4)
    })
 
    marker = " ← best" if f1_mac > best_f1 else ""
    print(f"  {w_r:>10.2f} {w_s:>10.2f} {f1_mac:>10.4f} {acc:>10.4f}{marker}")
 
    if f1_mac > best_f1:
        best_f1  = f1_mac
        best_w_r = w_r
        best_w_s = w_s
 
print(f"\n Trọng số tối ưu: w_resnet={best_w_r} | w_swin={best_w_s} | val F1={best_f1:.4f}")
 
if best_w_s > best_w_r:
    print(f"→ Trọng số nghiêng về Swin (w={best_w_s}) → Swin đóng vai trò"
          f" quan trọng hơn trong ensemble với bài toán này.")
elif best_w_r > best_w_s:
    print(f"→ Trọng số nghiêng về ResNet (w={best_w_r}) → ResNet đóng vai trò"
          f" quan trọng hơn trong ensemble với bài toán này.")
else:
    print(f"→ Hai mô hình đóng góp ngang nhau (w=0.5/0.5).")
 
df_weights = pd.DataFrame(weight_results)
df_weights.to_csv(os.path.join(OUTPUT_DIR, "weight_search.csv"), index=False)
print(f"Đã lưu: weight_search.csv")

In [ ]:
# ĐÁNH GIÁ ENSEMBLE TRÊN TẬP TEST
# Áp dụng đúng cùng chiến lược với val:
#   - Trọng số tối ưu từ grid search
#   - Adaptive weighting 0.3/0.7 cho nhóm lớp khó
 
# Ensemble cơ bản với trọng số tối ưu
prob_ensemble = best_w_r * resnet_test_probs + best_w_s * swin_test_probs
 
# Áp dụng adaptive weighting giống val nhất quán hoàn toàn
for cls in HARD_CLASSES:
    idx = CLASSES.index(cls)
    prob_ensemble[:, idx] = (
        0.3 * resnet_test_probs[:, idx] +
        0.7 * swin_test_probs[:, idx]
    )
 
ensemble_preds = prob_ensemble.argmax(axis=1)
ens_acc        = accuracy_score(test_labels, ensemble_preds)
ens_f1_mac     = f1_score(test_labels, ensemble_preds, average='macro', zero_division=0)
 
report_dict = classification_report(
    test_labels, ensemble_preds,
    target_names=CLASSES, digits=4, output_dict=True
)
 
print(f"\nKết quả Ensemble trên tập TEST (w_r={best_w_r}, w_s={best_w_s}):")
print(f"  Accuracy : {ens_acc:.4f}  ({ens_acc*100:.2f}%)")
print(f"  F1 Macro : {ens_f1_mac:.4f}")

In [ ]:
# XUẤT SỐ LIỆU
# 4 bảng:
#   1. Per-class Ensemble
#   2. So sánh 3 mô hình + Δ improvement
#   3. Phân tích nhóm khó MY+MMY+PMY
#   4. Summary JSON
 
# Classification report Ensemble
pd.DataFrame(report_dict).transpose().round(4).to_csv(
    os.path.join(OUTPUT_DIR, "ensemble_report.csv"))
print("\nĐã lưu: ensemble_report.csv")
 
# In bảng per-class Ensemble
print("\n" + "=" * 70)
print("BẢNG KẾT QUẢ PER-CLASS ENSEMBLE (copy vào luận văn)")
print("=" * 70)
print(f"  {'Lớp':<15} {'Precision':>10} {'Recall':>10} {'F1-score':>10} {'Support':>10}")
print(f"  {'-'*57}")
for cls in CLASSES:
    p  = report_dict[cls]["precision"]
    r  = report_dict[cls]["recall"]
    f1 = report_dict[cls]["f1-score"]
    s  = int(report_dict[cls]["support"])
    print(f"  {cls:<15} {p:>10.4f} {r:>10.4f} {f1:>10.4f} {s:>10}")
print(f"  {'-'*57}")
print(f"  {'Accuracy':<15} {'':>10} {'':>10} {ens_acc:>10.4f} {len(test_labels):>10}")
print(f"  {'Macro avg':<15} "
      f"{report_dict['macro avg']['precision']:>10.4f} "
      f"{report_dict['macro avg']['recall']:>10.4f} "
      f"{report_dict['macro avg']['f1-score']:>10.4f} "
      f"{len(test_labels):>10}")
print(f"  {'Weighted avg':<15} "
      f"{report_dict['weighted avg']['precision']:>10.4f} "
      f"{report_dict['weighted avg']['recall']:>10.4f} "
      f"{report_dict['weighted avg']['f1-score']:>10.4f} "
      f"{len(test_labels):>10}")
 
# Bảng so sánh 3 mô hình + Δ
print("\n" + "=" * 70)
print("SO SÁNH 3 MÔ HÌNH (bảng chính luận văn)")
print("=" * 70)
print(f"  {'Metric':<20} {'ResNet50':>10} {'Swin-S':>10} {'Ensemble':>10} "
      f"{'Δ (Ens-ResNet)':>16} {'Δ (Ens-Swin)':>14}")
print(f"  {'-'*75}")
for name, r_val, s_val, e_val in [
    ("Accuracy (%)",
     RESNET_RESULTS["accuracy"]*100,
     SWIN_RESULTS["accuracy"]*100,
     ens_acc*100),
    ("F1 Macro",
     RESNET_RESULTS["f1_macro"],
     SWIN_RESULTS["f1_macro"],
     ens_f1_mac),
]:
    print(f"  {name:<20} {r_val:>10.4f} {s_val:>10.4f} {e_val:>10.4f} "
          f"{e_val-r_val:>+16.4f} {e_val-s_val:>+14.4f}")
 
compare_data = []
for cls in CLASSES:
    r_f1 = RESNET_F1_PER_CLASS[cls]
    s_f1 = SWIN_F1_PER_CLASS[cls]
    e_f1 = report_dict[cls]["f1-score"]
    compare_data.append({
        "class"            : cls,
        "resnet50_f1"      : round(r_f1, 4),
        "swin_f1"          : round(s_f1, 4),
        "ensemble_f1"      : round(e_f1, 4),
        "delta_vs_resnet"  : round(e_f1 - r_f1, 4),
        "delta_vs_swin"    : round(e_f1 - s_f1, 4),
    })
pd.DataFrame(compare_data).to_csv(
    os.path.join(OUTPUT_DIR, "comparison_3models.csv"), index=False)
print(f"\nĐã lưu: comparison_3models.csv")
 
# Phân tích nhóm lớp khó
print("\n" + "=" * 70)
print("PHÂN TÍCH NHÓM LỚP KHÓ: MY + MMY + PMY")
print("(Insight quan trọng nhất — dùng trong phần kết luận luận văn)")
print("=" * 70)
print(f"  {'Lớp':<10} {'ResNet50':>10} {'Swin-S':>10} {'Ensemble':>10} "
      f"{'Δ Ens-ResNet':>14} {'Δ Ens-Swin':>12}")
print(f"  {'-'*62}")
 
hard_r_f1s = []
hard_s_f1s = []
hard_e_f1s = []
 
for cls in HARD_CLASSES:
    r_f1 = RESNET_F1_PER_CLASS[cls]
    s_f1 = SWIN_F1_PER_CLASS[cls]
    e_f1 = report_dict[cls]["f1-score"]
    hard_r_f1s.append(r_f1)
    hard_s_f1s.append(s_f1)
    hard_e_f1s.append(e_f1)
    print(f"  {cls:<10} {r_f1:>10.4f} {s_f1:>10.4f} {e_f1:>10.4f} "
          f"{e_f1-r_f1:>+14.4f} {e_f1-s_f1:>+12.4f}")
 
avg_r = np.mean(hard_r_f1s)
avg_s = np.mean(hard_s_f1s)
avg_e = np.mean(hard_e_f1s)
print(f"  {'-'*62}")
print(f"  {'Avg hard':<10} {avg_r:>10.4f} {avg_s:>10.4f} {avg_e:>10.4f} "
      f"{avg_e-avg_r:>+14.4f} {avg_e-avg_s:>+12.4f}")
print(f"\n→ Ensemble cải thiện nhóm khó trung bình "
      f"{(avg_e-avg_r)*100:+.2f}% so với ResNet50, "
      f"{(avg_e-avg_s)*100:+.2f}% so với Swin-S")
 
#Summary JSON
summary = {
    "ensemble_method"     : "Weighted Soft Voting + Adaptive Weighting",
    "best_w_resnet"       : best_w_r,
    "best_w_swin"         : best_w_s,
    "hard_class_w_resnet" : 0.3,
    "hard_class_w_swin"   : 0.7,
    "temperature_resnet"  : 1.2,
    "temperature_swin"    : 1.0,
    "test_accuracy"       : round(ens_acc, 4),
    "test_accuracy_pct"   : round(ens_acc * 100, 2),
    "test_f1_macro"       : round(ens_f1_mac, 4),
    "hard_classes_avg_f1" : round(avg_e, 4),
    "improvement_vs_resnet": {
        "accuracy_pct": round((ens_acc - RESNET_RESULTS["accuracy"]) * 100, 2),
        "f1_macro"    : round(ens_f1_mac - RESNET_RESULTS["f1_macro"], 4),
    },
    "improvement_vs_swin": {
        "accuracy_pct": round((ens_acc - SWIN_RESULTS["accuracy"]) * 100, 2),
        "f1_macro"    : round(ens_f1_mac - SWIN_RESULTS["f1_macro"], 4),
    },
}
with open(os.path.join(OUTPUT_DIR, "summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(f"\nĐã lưu: summary.json")

In [ ]:
# VẼ BIỂU ĐỒ
# 3 biểu đồ:
#   weight_search.png  
#   comparison_chart.png
#   confusion_matrix.png
 
# Biểu đồ 1: Grid search trọng số
fig, ax = plt.subplots(figsize=(10, 5))
w_swin_vals = [r["w_swin"]   for r in weight_results]
f1_vals     = [r["f1_macro"] for r in weight_results]
acc_vals    = [r["accuracy"] for r in weight_results]
 
ax.plot(w_swin_vals, f1_vals,  marker='o', color="#E74C3C",
        linewidth=2, label="F1 Macro (val)")
ax.plot(w_swin_vals, acc_vals, marker='s', color="#3498DB",
        linewidth=2, linestyle='--', label="Accuracy (val)")
ax.axvline(x=best_w_s, color="#27AE60", linestyle=':', linewidth=2,
           label=f"Tối ưu w_swin={best_w_s}")
ax.set_xlabel("Trọng số Swin (w_swin)", fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("Grid Search Trọng Số Ensemble trên Val Set",
             fontsize=13, fontweight='bold')
ax.set_xticks(w_swin_vals)
ax.tick_params(axis='x', rotation=45)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "weight_search.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Đã lưu: weight_search.png")
 
#Biểu đồ 2: So sánh F1 3 mô hình
resnet_f1s   = [RESNET_F1_PER_CLASS[c] for c in CLASSES]
swin_f1s     = [SWIN_F1_PER_CLASS[c]   for c in CLASSES]
ensemble_f1s = [report_dict[c]["f1-score"] for c in CLASSES]
 
x = np.arange(len(CLASSES))
w = 0.25
 
fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(x - w,   resnet_f1s,   w, label="ResNet50", color="#3498DB", alpha=0.85)
ax.bar(x,       swin_f1s,     w, label="Swin-S",   color="#E74C3C", alpha=0.85)
ax.bar(x + w,   ensemble_f1s, w, label="Ensemble", color="#27AE60", alpha=0.85)
 
for cls in HARD_CLASSES:
    idx = CLASSES.index(cls)
    ax.axvspan(idx - 0.45, idx + 0.45, alpha=0.08, color="#F39C12",
               label="Hard classes" if cls == HARD_CLASSES[0] else "")
 
ax.axhline(y=RESNET_RESULTS["f1_macro"], color="#3498DB", linestyle='--',
           alpha=0.5, linewidth=1.2,
           label=f"ResNet Macro ({RESNET_RESULTS['f1_macro']:.4f})")
ax.axhline(y=SWIN_RESULTS["f1_macro"],   color="#E74C3C", linestyle='--',
           alpha=0.5, linewidth=1.2,
           label=f"Swin Macro ({SWIN_RESULTS['f1_macro']:.4f})")
ax.axhline(y=ens_f1_mac, color="#27AE60", linestyle='--',
           alpha=0.5, linewidth=1.2,
           label=f"Ensemble Macro ({ens_f1_mac:.4f})")
 
ax.set_xlabel("Lớp tế bào", fontsize=12)
ax.set_ylabel("F1-score", fontsize=12)
ax.set_title("So sánh F1-score: ResNet50 vs Swin-S vs Ensemble",
             fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(CLASSES, rotation=30, ha='right')
ax.set_ylim(0.6, 1.08)
ax.legend(fontsize=9, ncol=2)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "comparison_chart.png"),
            dpi=150, bbox_inches='tight')
plt.show()
print("Đã lưu: comparison_chart.png")
 
# Biểu đồ 3: Confusion matrix Ensemble
cm      = confusion_matrix(test_labels, ensemble_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
 
fig, axes = plt.subplots(1, 2, figsize=(22, 9))
for ax, data, fmt, title in zip(
    axes, [cm, cm_norm], ["d", ".2f"],
    ["Confusion Matrix (Số lượng)", "Confusion Matrix (Tỉ lệ)"]
):
    sns.heatmap(data, annot=True, fmt=fmt,
                xticklabels=CLASSES, yticklabels=CLASSES,
                cmap="Blues", ax=ax, linewidths=0.5, annot_kws={"size": 8})
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("Actual", fontsize=11)
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)
 
plt.suptitle(
    f"Ensemble (w_r={best_w_r}, w_s={best_w_s}, T_r=1.2) — Confusion Matrix trên Test",
    fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"),
            dpi=150, bbox_inches='tight')
plt.show()
print("Đã lưu: confusion_matrix.png")

In [ ]:
# TỔNG KẾT CUỐI CÙNG
 
print("\n" + "=" * 70)
print("KẾT QUẢ CUỐI CÙNG — ENSEMBLE")
print("=" * 70)
print(f"  Phương pháp       : Weighted Soft Voting + Adaptive Weighting")
print(f"  Trọng số tối ưu   : ResNet={best_w_r} | Swin={best_w_s}")
print(f"  Adaptive (MY/MMY/PMY): ResNet=0.3 | Swin=0.7")
print(f"  Temperature       : ResNet T=1.2 | Swin T=1.0")
print(f"  Test Accuracy     : {ens_acc*100:.2f}%")
print(f"  Test F1 Macro     : {ens_f1_mac:.4f}")
print(f"\n  {'Model':<15} {'Accuracy':>10} {'F1 Macro':>10} {'Δ Acc':>10} {'Δ F1':>10}")
print(f"  {'-'*57}")
print(f"  {'ResNet50':<15} {RESNET_RESULTS['accuracy']*100:>9.2f}% "
      f"{RESNET_RESULTS['f1_macro']:>10.4f} {'—':>10} {'—':>10}")
print(f"  {'Swin-S':<15} {SWIN_RESULTS['accuracy']*100:>9.2f}% "
      f"{SWIN_RESULTS['f1_macro']:>10.4f} {'—':>10} {'—':>10}")
print(f"  {'Ensemble':<15} {ens_acc*100:>9.2f}% "
      f"{ens_f1_mac:>10.4f} "
      f"{(ens_acc-RESNET_RESULTS['accuracy'])*100:>+9.2f}% "
      f"{ens_f1_mac-RESNET_RESULTS['f1_macro']:>+10.4f}")
 
print(f"\n  Nhóm lớp khó (MY+MMY+PMY):")
print(f"    ResNet avg F1   : {avg_r:.4f}")
print(f"    Swin avg F1     : {avg_s:.4f}")
print(f"    Ensemble avg F1 : {avg_e:.4f}  ({(avg_e-avg_r)*100:+.2f}% vs ResNet)")
 
print(f"\n  File outputs:")
for fname, desc in {
    "weight_search.csv"     : "Kết quả grid search → bảng phân tích trọng số",
    "ensemble_report.csv"   : "Per-class Ensemble → bảng báo cáo",
    "comparison_3models.csv": "So sánh 3 mô hình → bảng chính luận văn",
    "summary.json"          : "Tóm tắt kết quả + cấu hình",
    "weight_search.png"     : "Biểu đồ grid search → hình báo cáo",
    "comparison_chart.png"  : "So sánh F1 3 mô hình → hình báo cáo",
    "confusion_matrix.png"  : "Ma trận nhầm lẫn → hình báo cáo",
}.items():
    fpath  = os.path.join(OUTPUT_DIR, fname)
    exists = "ok" if os.path.exists(fpath) else "ko ok"
    print(f"    {exists} {fname:<30} → {desc}")